In [2]:
%%bash
echo "🔧 Setting up environment for Stage 2 (Whisper & Qwen)..."
pip install -q faster-whisper "transformers==4.44.2" "tokenizers==0.19.1" accelerate
pip install -q fastapi uvicorn python-multipart pyngrok pydantic
echo "✅ Environment Setup Complete!"

🔧 Setting up environment for Stage 2 (Whisper & Qwen)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 29.8 MB/s eta 0:00:00
✅ Environment Setup Complete!


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%writefile server_stage2_llm.py
import os
import re
import requests
import torch
from pydantic import BaseModel
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from faster_whisper import WhisperModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import uvicorn
from pyngrok import ngrok

PROJECT_DIR = '/content/drive/MyDrive/Video_Translation_Project'
MODELS_CACHE = os.path.join(PROJECT_DIR, 'Models_Cache')
os.environ['HF_HOME'] = MODELS_CACHE
os.environ['XDG_DATA_HOME'] = MODELS_CACHE

device = "cuda" if torch.cuda.is_available() else "cpu"
app = FastAPI(title="Stage 2: LLM Translation API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

print("[INFO] Loading Whisper (STT)...")
try:
    stt_model = WhisperModel("large-v3", device=device, compute_type="float16")
except Exception as e:
    print(f"[WARNING] float16 failed, falling back to int8. Error: {e}")
    stt_model = WhisperModel("large-v3", device=device, compute_type="int8")

print("[INFO] Loading Qwen (LLM) from custom Drive path...")

LLM_DRIVE_PATH = "/content/drive/MyDrive/Video_Translation_Project/Qwen_Model_Files"

tokenizer = AutoTokenizer.from_pretrained(LLM_DRIVE_PATH)
llm_model = AutoModelForCausalLM.from_pretrained(LLM_DRIVE_PATH, torch_dtype=torch.float16, device_map="auto")
print("[INFO] Models loaded successfully!")

class AudioRequest(BaseModel):
    vocals_url: str

@app.post("/process_stage2_from_url")
async def process_stage2_from_url(payload: AudioRequest):
    input_audio = "/content/local_vocals.wav"

    print("[INFO] Downloading clean vocals securely...")
    res = requests.get(payload.vocals_url, headers={'ngrok-skip-browser-warning': 'true'})
    with open(input_audio, 'wb') as f:
        f.write(res.content)

    print("[INFO] Transcribing...")
    segments, _ = stt_model.transcribe(input_audio, beam_size=5, language="en")
    english_text = " ".join([segment.text for segment in segments])

    print("[INFO] Translating...")
    system_prompt = "You are a translation engine. Translate the English text to Arabic. Output ONLY the Arabic text. No Chinese. No explanations."
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": f"English: {english_text}\nArabic:"}]
    text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    model_inputs = tokenizer([text_input], return_tensors="pt").to(device)


    generated_ids = llm_model.generate(
        input_ids=model_inputs.input_ids,
        attention_mask=model_inputs.attention_mask,
        max_new_tokens=1024,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.15
    )

    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    arabic_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

    arabic_text = re.sub(r'[\u4e00-\u9fff]+', '', arabic_text).strip()


    arabic_text = re.sub(r'\s+', ' ', arabic_text)

    print("[INFO] Stage 2 Complete!")
    return {"english_text": english_text, "arabic_text": arabic_text}

if __name__ == "__main__":
    ngrok.set_auth_token("")
    public_url = ngrok.connect(8002).public_url
    print(f"\n🚀 STAGE 2 MICROSERVICE IS LIVE AT: {public_url}\n")
    uvicorn.run(app, host="0.0.0.0", port=8002)

Writing server_stage2_llm.py


In [ ]:
!python3 server_stage2_llm.py

[INFO] Loading Whisper (STT)...
preprocessor_config.json: 100% 340/340 [00:00<00:00, 2.36MB/s]
tokenizer.json: 0.00B [00:00, ?B/s]
config.json: 2.39kB [00:00, 3.11MB/s]

vocabulary.json: 1.07MB [00:00, 34.1MB/s]
tokenizer.json: 2.48MB [00:00, 44.8MB/s]
model.bin: 100% 3.09G/3.09G [00:25<00:00, 122MB/s]
[INFO] Loading Qwen (LLM) from custom Drive path...
[INFO] Models loaded successfully!

🚀 STAGE 2 MICROSERVICE IS LIVE AT: https://anteater-purchase-turtle.ngrok-free.dev

INFO:     Started server process [1349]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8002 (Press CTRL+C to quit)
INFO:     197.63.240.96:0 - "OPTIONS /process_stage2_from_url HTTP/1.1" 200 OK
[INFO] Downloading clean vocals securely...
[INFO] Transcribing...
[INFO] Translating...
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is